# 03 — TTA on CIFAR-10-C: STE vs Gaussian surrogate (INT4 first)
Inputs: `qtta-code` dataset (qtta_lib.py + resnet20_fp32.pt), CIFAR-10 Python, and a CIFAR-10-C dataset (`gaussian_noise.npy`, ..., `labels.npy`).
Protocol: episodic per corruption (fresh model each), online within it, bs=64, 1 Adam step/batch, BN affine only, entropy loss. Accuracy = predictions from the forward pass before each update (TENT convention).

In [ ]:
import sys, os, glob, copy, time
import numpy as np, pandas as pd, torch, torch.nn as nn
sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/qtta_lib.py", recursive=True)[0]))
import qtta_lib as Q
from qtta_lib import QCFG
dev = "cuda" if torch.cuda.is_available() else "cpu"

CORR = ["gaussian_noise", "defocus_blur", "fog", "contrast", "jpeg_compression"]
SEV = 5
MEAN = torch.tensor(Q.MEAN).view(1, 3, 1, 1); STD = torch.tensor(Q.STD).view(1, 3, 1, 1)

def load_c(name, sev=SEV):
    path = glob.glob(f"/kaggle/input/**/{name}.npy", recursive=True)[0]
    lab = os.path.join(os.path.dirname(path), "labels.npy")
    if not os.path.exists(lab): lab = glob.glob("/kaggle/input/**/labels.npy", recursive=True)[0]
    sl = slice((sev - 1) * 10000, sev * 10000)
    x = torch.from_numpy(np.ascontiguousarray(np.load(path, mmap_mode="r")[sl])).permute(0, 3, 1, 2).float() / 255
    y = torch.from_numpy(np.load(lab)[sl]).long()
    return ((x - MEAN) / STD).to(dev), y.to(dev)

DATA = {c: load_c(c) for c in CORR}
print({c: tuple(v[0].shape) for c, v in DATA.items()})

## Models

In [ ]:
fp, ck = Q.load_fp32(Q.find_file("resnet20_fp32.pt"), dev)
train_loader, test_loader = Q.get_cifar10(bs=500)
base = {"FP32": fp}
for name, (wb, ab) in {"W8A8": (8, 8), "W4A4": (4, 4)}.items():
    base[name], _ = Q.quantize_model(fp, wb, ab, train_loader, dev)
    print(name, "clean:", round(Q.evaluate(base[name], test_loader, dev), 2))

## TTA machinery

In [ ]:
def entropy(lg): return -(lg.softmax(1) * lg.log_softmax(1)).sum(1).mean()

def configure(m):
    m.train()
    for p in m.parameters(): p.requires_grad_(False)
    params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.track_running_stats = False; mod.running_mean = None; mod.running_var = None
            mod.weight.requires_grad_(True); mod.bias.requires_grad_(True)
            params += [mod.weight, mod.bias]
    return params

def run_tta(model, x, y, mode="ste", sigma=0.5, lr=1e-3, bs=64, seed=0, adapt=True):
    m = copy.deepcopy(model); params = configure(m)
    QCFG.mode, QCFG.sigma = mode, sigma
    opt = torch.optim.Adam(params, lr=lr) if adapt else None
    perm = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed)).to(dev)
    correct = 0
    if dev == "cuda": torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.time()
    with torch.set_grad_enabled(adapt):
        for i in range(0, len(x), bs):
            idx = perm[i:i + bs]
            logits = m(x[idx])
            if adapt:
                opt.zero_grad(set_to_none=True); entropy(logits).backward(); opt.step()
            correct += (logits.argmax(1) == y[idx]).sum().item()
    if dev == "cuda": torch.cuda.synchronize()
    mem = torch.cuda.max_memory_allocated() / 2**20 if dev == "cuda" else 0
    return 100 * correct / len(x), time.time() - t0, mem

@torch.no_grad()
def static_acc(model, x, y):
    m = copy.deepcopy(model).eval()
    return sum((m(x[i:i + 500]).argmax(1) == y[i:i + 500]).sum().item() for i in range(0, len(x), 500)) / len(x) * 100

## Headroom check: no-TTA accuracy under corruption, and BN-statistics-only adaptation

In [ ]:
rows = []
for c in CORR:
    x, y = DATA[c]
    r = {"corruption": c}
    for n in ["FP32", "W8A8", "W4A4"]: r[f"{n} no-TTA"] = static_acc(base[n], x, y)
    r["W4A4 BN-stats only"] = run_tta(base["W4A4"], x, y, adapt=False)[0]
    rows.append(r)
head = pd.DataFrame(rows).set_index("corruption"); head.loc["mean"] = head.mean()
print(head.round(2).to_string())

## Sweep: STE vs Gaussian (fixed sigma/Delta), W4A4, 3 seeds

In [ ]:
CONFIGS = [("STE", "ste", 0.0)] + [(f"gauss_{s}", "gauss", s) for s in [0.15, 0.2, 0.25, 0.35, 0.5, 1.0]]
SEEDS = [0, 1, 2]
res = []
for c in CORR:
    x, y = DATA[c]
    for name, mode, s in CONFIGS:
        for seed in SEEDS:
            acc, t, mem = run_tta(base["W4A4"], x, y, mode, s, seed=seed)
            res.append(dict(corruption=c, method=name, seed=seed, acc=acc, time_s=t, mem_mb=mem))
    print("done", c)
df = pd.DataFrame(res); df.to_csv("/kaggle/working/tta_w4a4_fixed_sigma.csv", index=False)

In [ ]:
per_seed = df.groupby(["method", "seed"]).acc.mean().reset_index()          # mean over corruptions, per seed
summ = per_seed.groupby("method").acc.agg(["mean", "std"])
summ["delta_vs_STE"] = summ["mean"] - summ.loc["STE", "mean"]
summ["time_s/run"] = df.groupby("method").time_s.mean()
print(summ.loc[[n for n, _, _ in CONFIGS]].round(3).to_string())
print()
print(df.pivot_table(index="method", columns="corruption", values="acc").loc[[n for n, _, _ in CONFIGS]].round(2).to_string())